# Differentiable Monte Carlo — The Real Deal

**Goal:** Convert the paper's MC simulation into a fully differentiable pipeline in PyTorch.

**Plan (step by step):**
1. Implement the full MC simulation as a PyTorch module
2. Make the sampling differentiable (from toy-experiment-sampling)
3. Make the Voigt fit differentiable via implicit differentiation (from toy-experiment-fitting)
4. Compare simulated vs experimental distributions using smooth density comparison
5. Optimize (γ, n̄) via gradient descent

---

## Step 1: Imports & Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass

print('Ready')

## Step 2: One MC Run (Continuous)

One run = simulate one PLE scan with **continuous photon positions** (no binning).

Steps:
1. Sample total photon count $n \sim \mathcal{N}(\bar{n}, \sigma)$
2. Draw $n$ photon frequencies from the Cauchy($\gamma$) lineshape (continuous)
3. Add background noise: draw $n_{\text{bg}} \sim \text{Poisson}(\lambda)$, distribute uniformly
4. **Return:** all photon frequencies concatenated into one tensor

Later: fit a Voigt to these continuous samples to extract one linewidth $w_i$.

In [ ]:
@dataclass
class MCParams:
    gamma: float       # HWHM of Cauchy (MHz) — optimized
    nbar: float        # mean photon count — optimized
    sigma: float = 6.0  # noise std (fixed, from paper)
    lambda_: float = 2.0  # mean background counts (fixed, from paper)

# Frequency window (from paper)
FREQ_MIN = -75.0   # MHz
FREQ_MAX = 75.0    # MHz
print(f'Frequency window: [{FREQ_MIN}, {FREQ_MAX}] MHz')

In [ ]:
def sample_photon_count(nbar, sigma, epsilon):
    """Reparameterized sampling of total photon count (non-negative int)."""
    n_float = nbar + sigma * epsilon
    return max(round(n_float), 0)

def one_run(params, epsilon, rng=None):
    """
    One MC run simulating one PLE scan.
    
    Returns:
        signal_freqs: tensor of shape (n,) — signal photon frequencies
        bg_freqs: tensor of shape (n_bg,) — background photon frequencies
        (Concatenated they form the full "measured" spectrum as point cloud)
    """
    if rng is None:
        rng = np.random.default_rng()
    
    # 1. Sample total signal photon count
    n = sample_photon_count(params.nbar, params.sigma, epsilon)
    
    # 2. Sample n continuous photon frequencies from Cauchy(gamma)
    #    Cauchy can be sampled via: gamma * tan(pi*(u - 0.5)) where u ~ Uniform(0,1)
    if n > 0:
        u = rng.uniform(0, 1, n)
        samples = params.gamma * np.tan(np.pi * (u - 0.5))
        # Clip to frequency window
        signal_freqs = np.clip(samples, FREQ_MIN, FREQ_MAX)
    else:
        signal_freqs = np.array([])
    
    # 3. Background noise: uniform across window
    n_bg = rng.poisson(params.lambda_)
    if n_bg > 0:
        bg_freqs = rng.uniform(FREQ_MIN, FREQ_MAX, n_bg)
    else:
        bg_freqs = np.array([])
    
    all_freqs = torch.cat([
        torch.tensor(signal_freqs, dtype=torch.float32),
        torch.tensor(bg_freqs, dtype=torch.float32),
    ])
    return all_freqs


# Quick test
params = MCParams(gamma=15.0, nbar=40.0)
eps = np.random.normal()
photons = one_run(params, eps)

print(f'Sample eps = {eps:.2f}')
print(f'Total photons: {len(photons)}  (nbar={params.nbar})')
print(f'Background events: ~Poisson(lambda={params.lambda_})')
print(f'Freq range: [{photons.min().item():.1f}, {photons.max().item():.1f}] MHz')
print(f'First 10 photons (freqs): {photons[:10].tolist()}')

The key change: no bins, no histogram. Each run gives us a **point cloud** of detected photon frequencies.

Next up: **fitting a Voigt** to each run's point cloud to extract the FWHM $w_i$.

## Step 3: Fitting a Voigt to the Photon Point Cloud

Each run gives us a set of photon frequencies. We extract one FWHM by fitting a
pseudo-Voigt profile via **maximum likelihood estimation (MLE)**:

1. Define the pseudo-Voigt PDF: $V(f) = \eta \cdot G(f) + (1-\eta) \cdot L(f)$
2. Neg log-likelihood = $-\sum \log V(\text{photon\_freq})$  
3. Optimize (center, $\gamma$, $\sigma_G$, $\eta$) via L-BFGS
4. Return FWHM from the fitted pseudo-Voigt

Currently non-differentiable (torch.no_grad). We'll make it differentiable later with implicit differentiation.

In [ ]:
def pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta):
    """Log-PDF of pseudo-Voigt. All params in unconstrained space for stable opt."""
    gamma = torch.exp(log_gamma)
    sigma_g = torch.exp(log_sigma_g)
    eta = torch.sigmoid(logit_eta)
    
    # Gaussian part
    gauss = torch.exp(-0.5 * ((freqs - center) / sigma_g) ** 2)
    gauss = gauss / (sigma_g * torch.sqrt(torch.tensor(2.0 * torch.pi)))
    
    # Lorentzian part
    lorentz = (gamma / torch.pi) / ((freqs - center) ** 2 + gamma ** 2)
    
    pdf = eta * gauss + (1 - eta) * lorentz
    return torch.log(pdf + 1e-30)  # avoid log(0)

def fit_pseudo_voigt(photons, n_iters=200):
    """
    Fit a pseudo-Voigt to photon frequencies via MLE (L-BFGS).
    
    Returns:
        (fwhm, params_dict) where fwhm is the FWHM from the fit
    """
    if len(photons) < 3:
        # Too few photons — return a default
        return 50.0, None
    
    freqs = photons.clone().detach().float()
    
    # Initialize params: center at median, gamma ~ 15, sigma_g ~ 5, eta ~ 0.5
    center = torch.tensor(float(freqs.median()), requires_grad=True)
    log_gamma = torch.tensor(np.log(15.0), requires_grad=True)
    log_sigma_g = torch.tensor(np.log(5.0), requires_grad=True)
    logit_eta = torch.tensor(0.0, requires_grad=True)  # sigmoid(0) = 0.5
    
    optimizer = torch.optim.LBFGS([center, log_gamma, log_sigma_g, logit_eta],
                                   max_iter=n_iters, line_search_fn='strong_wolfe')
    
    def closure():
        optimizer.zero_grad()
        log_pdf = pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta)
        nll = -log_pdf.mean()
        nll.backward()
        return nll
    
    optimizer.step(closure)
    
    gamma = torch.exp(log_gamma).item()
    sigma_g = torch.exp(log_sigma_g).item()
    fwhm = 2.0 * gamma  # Approximate: use Lorentzian FWHM as primary width
    # (A full Voigt FWHM would need the approximation, but this is fine for now)
    
    return fwhm, {
        'center': center.item(),
        'gamma': gamma,
        'sigma_g': sigma_g,
        'eta': torch.sigmoid(logit_eta).item(),
        'fwhm': fwhm,
    }


# Test: generate photons from known params, recover FWHM
torch.manual_seed(42)
true_gamma = 15.0

# Generate clean Cauchy samples (no background)
u = torch.rand(200)
true_photons = true_gamma * torch.tan(torch.pi * (u - 0.5))
true_photons = torch.clamp(true_photons, FREQ_MIN, FREQ_MAX)

fwhm, params = fit_pseudo_voigt(true_photons)
print(f'True gamma = {true_gamma} MHz  →  true FWHM = {2*true_gamma} MHz')
print(f'Fit  gamma = {params["gamma"]:.2f} MHz  →  fit FWHM = {fwhm:.2f} MHz')
print(f'Fit: center={params["center"]:.2f}, sigma_g={params["sigma_g"]:.2f}, eta={params["eta"]:.2f}')